# 🤖 Support Vector Machine — Concepts
### Atlas Skilltech University — Machine Learning Course
---
**Is notebook mein hum sikhenge:**
- SVM kya hai aur kyun use karte hain
- Hyperplane aur Margins ka concept
- Support Vectors kya hote hain
- Kernel Trick aur uska mathematics
- SVM Classification aur Regression implementation

**Real World Analogy:**
> Socho tumhare paas ek table hai jisme Red aur Blue marbles hain.
> SVM ek aisi line/surface dhundta hai jo dono ko **best possible tarike se alag kare**.
> Yahi SVM ka kaam hai!

## Step 1: Libraries Import Karo
---
**Keyword Explanation:**
- **SVC** — Support Vector Classifier (Classification ke liye)
- **SVR** — Support Vector Regressor (Regression ke liye)
- **make_classification** — sample classification data banana
- **make_blobs** — cluster-shaped sample data banana

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC, SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score
)
from sklearn.datasets import make_classification, make_blobs, make_circles, make_moons

import warnings
warnings.filterwarnings('ignore')

print("✅ Saari libraries successfully import ho gayi!")

## Part 1: SVM kya hota hai? (Introduction)
---
**SVM = Support Vector Machine**

**Simple Explanation:**
> Ek aisi line (ya surface) dhundho jo **dono classes ke beech maximum gap (margin) rakhey**.
> Yeh line itni achi jagah ho ki naye data points bhi sahi classify ho sakein.

**Real World Examples:**
- 🏥 Cancer cells vs Normal cells classify karna
- 📧 Spam vs Not Spam emails
- 🖼️ Image mein cat hai ya dog?
- 💳 Credit card fraud detection

**Keyword:**
- **Supervised Learning** — labeled data chahiye (hum jaante hain answer)
- **Classification + Regression** — dono kaam kar sakta hai
- **Maximum Margin** — SVM ki specialty — best possible separation

In [ ]:
# Simple 2D data banao — do classes
np.random.seed(42)

# Class 0: Blue team (Setosa jaisi)
class0 = np.random.randn(30, 2) + np.array([2, 2])
# Class 1: Red team (Virginica jaisi)
class1 = np.random.randn(30, 2) + np.array([6, 6])

X_simple = np.vstack([class0, class1])
y_simple = np.array([0]*30 + [1]*30)

# Visualize
plt.figure(figsize=(8, 6))
plt.scatter(class0[:, 0], class0[:, 1], c='blue', s=60, label='Class 0 (Blue Team)', edgecolor='black')
plt.scatter(class1[:, 0], class1[:, 1], c='red', s=60, label='Class 1 (Red Team)', edgecolor='black')
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('SVM ka Problem: Dono Classes ko Alag Karo!', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()
print("Dekho: Dono classes clearly alag hain — SVM inhe best possible line se divide karega!")

## Part 2: Hyperplane aur Margins
---
**Hyperplane kya hota hai?**
> - 2D mein: ek line (`ax + by + c = 0`)
> - 3D mein: ek plane
> - nD mein: ek (n-1) dimensional surface
>
> Yeh woh surface hai jo dono classes ko divide karti hai.

**Margin kya hota hai?**
> Hyperplane ke dono taraf ek **buffer zone** hoti hai.
> SVM yeh margin **maximize** karna chahta hai.
> Zyada margin = zyada confident classification = better generalization

**Support Vectors kya hote hain?**
> Woh data points jo margin ke **boundary par** hain.
> Yahi points SVM ke "support" karte hain — isliye naam Support Vector Machine!
> Agar yeh points hatao, to hyperplane badal jaata hai.

**Keyword:**
- **Decision Boundary** — woh line jo predict karne ke liye use hoti hai
- **Margin** — decision boundary ke dono taraf space
- **Maximum Margin Classifier** — SVM ka goal: margin maximize karna

In [ ]:
# SVM train karo aur decision boundary dikhao
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_simple)

svm_linear = SVC(kernel='linear', C=1.0)
svm_linear.fit(X_scaled, y_simple)

# Decision boundary plot function
def plot_svm_boundary(model, X, y, title, ax):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')
    ax.scatter(X[y==0, 0], X[y==0, 1], c='blue', s=60, edgecolor='black', label='Class 0')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='red', s=60, edgecolor='black', label='Class 1')
    
    # Support vectors highlight karo
    ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
               s=200, linewidth=2, facecolors='none', edgecolors='green', label='Support Vectors')
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig, ax = plt.subplots(figsize=(8, 6))
plot_svm_boundary(svm_linear, X_scaled, y_simple, 'SVM — Decision Boundary + Support Vectors', ax)
plt.tight_layout()
plt.show()

print(f"Support Vectors ki sankhya: {len(svm_linear.support_vectors_)}")
print("Green circles = Support Vectors (yeh points margin ko define karte hain!)")

In [ ]:
# C parameter ka asar — Hard Margin vs Soft Margin
# C = Regularization parameter
# C bada = kam tolerance for misclassification (Hard Margin)
# C chota = zyada tolerance for misclassification (Soft Margin)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

c_values = [0.01, 1.0, 100.0]
titles = ['C = 0.01\n(Soft Margin - Zyada Tolerant)', 
          'C = 1.0\n(Balance)', 
          'C = 100\n(Hard Margin - Kam Tolerant)']

for ax, C, title in zip(axes, c_values, titles):
    svm_c = SVC(kernel='linear', C=C)
    svm_c.fit(X_scaled, y_simple)
    plot_svm_boundary(svm_c, X_scaled, y_simple, title, ax)
    ax.set_xlabel(f'Support Vectors: {len(svm_c.support_vectors_)}')

plt.suptitle('C Parameter ka Asar — Hard vs Soft Margin', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Observation: C bada hoga to margin chhota hoga aur overfitting ka risk badh jaata hai!")

## Part 3: Kernel Trick — Jab Data Linearly Separable Na Ho
---
**Problem:**
> Kya hoga jab data ko ek straight line se separate nahi kar sakte?
> Jaise: Circle ke andar ek class, bahar doosri class.

**Kernel Trick ka Idea:**
> Data ko **higher dimension mein lift karo**!
> 2D mein jo separate nahi ho raha, 3D mein ho sakta hai.
>
> Analogy: Maan lo floor par red aur blue marbles hain mixed.
> Agar table ko thoda hila do (z dimension add karo), to blue wale upar aayen aur red wale neeche rahein.
> Ab ek flat surface se alag kar sakte hain!

**Kernel Functions:**
| Kernel | Use Case | Formula |
|--------|----------|----------|
| **Linear** | Linearly separable data | K(x,y) = x·y |
| **RBF (Gaussian)** | Most common, non-linear | K(x,y) = exp(-γ||x-y||²) |
| **Polynomial** | Polynomial boundaries | K(x,y) = (x·y + r)^d |
| **Sigmoid** | Neural network jaisa | K(x,y) = tanh(αx·y + c) |

**Keyword:**
- **Kernel** — ek function jo dot product calculate karta hai higher dimensional space mein, without actually going there!
- **Gamma (γ)** — RBF kernel ka parameter — controls curve ki smoothness
- **Degree (d)** — Polynomial kernel ka parameter — polynomial ki power

In [ ]:
# Non-linear data — circles
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)
X_moons, y_moons = make_moons(n_samples=200, noise=0.15, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_circles[y_circles==0, 0], X_circles[y_circles==0, 1], 
                c='blue', s=40, edgecolor='black', label='Class 0')
axes[0].scatter(X_circles[y_circles==1, 0], X_circles[y_circles==1, 1], 
                c='red', s=40, edgecolor='black', label='Class 1')
axes[0].set_title('Circle Data — Linear SVM Fail Karega!', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_moons[y_moons==0, 0], X_moons[y_moons==0, 1], 
                c='blue', s=40, edgecolor='black', label='Class 0')
axes[1].scatter(X_moons[y_moons==1, 0], X_moons[y_moons==1, 1], 
                c='red', s=40, edgecolor='black', label='Class 1')
axes[1].set_title('Moon Data — Linear SVM Fail Karega!', fontsize=11, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Non-Linear Data — Kernel Trick ki Zaroorat!', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Alag-alag Kernels compare karo — Circle data par
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
titles = ['Linear Kernel\n(Fail karega)', 'RBF Kernel\n(Best!)', 
          'Polynomial Kernel', 'Sigmoid Kernel']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, kernel, title in zip(axes, kernels, titles):
    svm_k = SVC(kernel=kernel, C=1.0, gamma='scale')
    svm_k.fit(X_circles, y_circles)
    acc = svm_k.score(X_circles, y_circles)
    
    # Decision boundary
    h = 0.02
    x_min, x_max = X_circles[:, 0].min() - 0.3, X_circles[:, 0].max() + 0.3
    y_min, y_max = X_circles[:, 1].min() - 0.3, X_circles[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svm_k.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_circles[y_circles==0, 0], X_circles[y_circles==0, 1], c='blue', s=30, edgecolor='k')
    ax.scatter(X_circles[y_circles==1, 0], X_circles[y_circles==1, 1], c='red', s=30, edgecolor='k')
    ax.set_title(f'{title}\nAcc: {acc:.2f}', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('4 Kernels Compare — Circle Data Par', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("RBF Kernel ne best kaam kiya! Circle data ko perfectly separate kar diya.")

## Part 4: Kernel Mathematics
---
**Linear Kernel:**
```
K(x, y) = x · y = Σ xi * yi
```
Direct dot product — sirf linear boundaries ke liye.

**RBF (Radial Basis Function) Kernel — Sabse Common:**
```
K(x, y) = exp(-γ * ||x - y||²)
```
- γ (gamma) control karta hai kitna influence ek point ka hoga
- γ bada = zyada curved boundary (overfit risk)
- γ chota = zyada smooth boundary (underfit risk)

**Polynomial Kernel:**
```
K(x, y) = (γ * x·y + r)^d
```
- d = degree (order of polynomial)
- r = coef0 parameter

**Keyword:**
- **Gamma (γ)** — RBF/Poly kernel ka bandwidth parameter
- **Degree** — Polynomial kernel ki power
- **C** — Regularization (tradeoff between margin aur misclassification)

In [ ]:
# Gamma ka asar — RBF kernel par
X_circ_scaled = StandardScaler().fit_transform(X_circles)

gamma_values = [0.01, 0.1, 1.0, 10.0]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, gamma in zip(axes, gamma_values):
    svm_g = SVC(kernel='rbf', C=1.0, gamma=gamma)
    svm_g.fit(X_circ_scaled, y_circles)
    acc = svm_g.score(X_circ_scaled, y_circles)
    
    h = 0.02
    x_min, x_max = X_circ_scaled[:, 0].min() - 0.3, X_circ_scaled[:, 0].max() + 0.3
    y_min, y_max = X_circ_scaled[:, 1].min() - 0.3, X_circ_scaled[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svm_g.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_circ_scaled[y_circles==0, 0], X_circ_scaled[y_circles==0, 1], 
               c='blue', s=30, edgecolor='k')
    ax.scatter(X_circ_scaled[y_circles==1, 0], X_circ_scaled[y_circles==1, 1], 
               c='red', s=30, edgecolor='k')
    ax.set_title(f'Gamma = {gamma}\nAcc = {acc:.2f}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Gamma Parameter ka Asar — RBF Kernel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Gamma = 10 mein boundary bahut jagged hai — overfitting!")
print("Gamma = 0.01 mein boundary bahut smooth hai — underfitting!")

## Part 5: SVM Classification — Practical Example
---
**Problem: Student Pass/Fail Predict Karo**

Features:
- Study Hours (Padhne ka samay)
- Sleep Hours (Neend)
- Practice Tests (Practice ke test)

Target: Pass (1) ya Fail (0)

In [ ]:
# Student dataset banao
np.random.seed(42)
n = 150

study_hours = np.random.uniform(1, 10, n)
sleep_hours = np.random.uniform(4, 9, n)
practice_tests = np.random.randint(0, 10, n)

# Pass/Fail logic
score = 0.5*study_hours + 0.3*sleep_hours + 0.2*practice_tests + np.random.randn(n)*0.5
passed = (score > score.mean()).astype(int)

df_students = pd.DataFrame({
    'Study_Hours': study_hours.round(1),
    'Sleep_Hours': sleep_hours.round(1),
    'Practice_Tests': practice_tests,
    'Result': passed
})
df_students['Result_Label'] = df_students['Result'].map({0: 'Fail', 1: 'Pass'})

print("Student Dataset:")
print(df_students.head(10))
print(f"\nTotal students: {n}")
print(f"Pass: {passed.sum()} | Fail: {(1-passed).sum()}")

In [ ]:
# Features aur Target
X_students = df_students[['Study_Hours', 'Sleep_Hours', 'Practice_Tests']].values
y_students = df_students['Result'].values

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_students, y_students, test_size=0.2, random_state=42, stratify=y_students
)

# Scaling
scaler_s = StandardScaler()
X_train_s = scaler_s.fit_transform(X_train)
X_test_s = scaler_s.transform(X_test)

# SVM Model — RBF Kernel
svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_clf.fit(X_train_s, y_train)

y_pred = svm_clf.predict(X_test_s)
print(f"Training Accuracy: {svm_clf.score(X_train_s, y_train)*100:.2f}%")
print(f"Testing Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Support Vectors:   {len(svm_clf.support_vectors_)}")

In [ ]:
# Confusion Matrix — dono methods
cm = confusion_matrix(y_test, y_pred)
labels = ['Fail', 'Pass']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Purana tarika
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            annot_kws={'size': 14}, ax=axes[0])
axes[0].set_title('Purana Tarika: Seaborn Heatmap', fontweight='bold')
axes[0].set_xlabel('Predicted (Model ne kya bola)')
axes[0].set_ylabel('Actual (Sach mein kya tha)')

# Naya tarika
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
cmd.plot(cmap='Greens', colorbar=False, ax=axes[1])
axes[1].set_title('Naya Tarika: ConfusionMatrixDisplay', fontweight='bold')

plt.suptitle('SVM Classification — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=labels))

In [ ]:
# Alag Kernels compare karo — Student Data
print("Kernel Comparison — Student Pass/Fail Dataset")
print("-" * 55)

kernel_configs = [
    ('linear', {}, 'Linear'),
    ('rbf', {'gamma': 'scale'}, 'RBF (default gamma)'),
    ('rbf', {'gamma': 0.1}, 'RBF (gamma=0.1)'),
    ('poly', {'degree': 2}, 'Polynomial (degree=2)'),
    ('poly', {'degree': 3}, 'Polynomial (degree=3)'),
]

print(f"{'Kernel':<30} {'Train Acc':<12} {'Test Acc':<12}")
print("-" * 55)

for kernel, params, name in kernel_configs:
    svm_temp = SVC(kernel=kernel, C=1.0, **params)
    svm_temp.fit(X_train_s, y_train)
    train_acc = svm_temp.score(X_train_s, y_train)
    test_acc = svm_temp.score(X_test_s, y_test)
    print(f"{name:<30} {train_acc:.4f}      {test_acc:.4f}")

## Part 6: SVM Regression (SVR)
---
**SVR kya hai?**
> SVM ka regression version. Classification mein hum classes separate karte the,
> Regression mein hum ek **epsilon tube** ke andar fit karne ki koshish karte hain.

**Epsilon Tube:**
> Hyperplane ke dono taraf ek tube banao (epsilon = ε).
> Jo points tube ke andar hain — no penalty (thoda error allowed).
> Jo points tube ke bahar hain — penalize karo.

**Real World Use:**
- 🏠 House price prediction
- 📈 Stock price forecasting
- 🌡️ Temperature prediction

**Keyword:**
- **Epsilon (ε)** — tube ki width — kitna error tolerable hai
- **SVR** — Support Vector Regressor

In [ ]:
# SVR — House Price Prediction (sample data)
np.random.seed(42)
n_houses = 100

size = np.random.uniform(500, 3000, n_houses)   # Ghar ka size (sqft)
price = 50 * size + 10000 * np.random.randn(n_houses) + 50000  # Price

X_house = size.reshape(-1, 1)
y_house = price

# Scale karo
scaler_h = StandardScaler()
X_house_s = scaler_h.fit_transform(X_house)

# SVR model
svr_model = SVR(kernel='rbf', C=100, epsilon=0.1)
svr_model.fit(X_house_s, y_house)

# Predictions
X_plot = np.linspace(500, 3000, 300).reshape(-1, 1)
X_plot_s = scaler_h.transform(X_plot)
y_plot = svr_model.predict(X_plot_s)

plt.figure(figsize=(10, 6))
plt.scatter(size, price, c='steelblue', s=40, alpha=0.6, label='Actual House Prices', edgecolor='k')
plt.plot(X_plot.flatten(), y_plot, 'r-', linewidth=2.5, label='SVR Prediction')
plt.fill_between(X_plot.flatten(), 
                  y_plot - 15000, y_plot + 15000, 
                  alpha=0.2, color='red', label='Epsilon Tube (±15000)')
plt.xlabel('Ghar ka Size (sqft)', fontsize=12)
plt.ylabel('Price (Rs)', fontsize=12)
plt.title('SVR — House Price Prediction + Epsilon Tube', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

from sklearn.metrics import mean_squared_error, r2_score
y_pred_house = svr_model.predict(X_house_s)
print(f"RMSE: {np.sqrt(mean_squared_error(y_house, y_pred_house)):.2f}")
print(f"R2 Score: {r2_score(y_house, y_pred_house):.4f}")

In [ ]:
# SVR — Alag Kernels compare karo
from sklearn.metrics import r2_score

svr_kernels = [
    SVR(kernel='linear', C=100),
    SVR(kernel='rbf', C=100, epsilon=0.1),
    SVR(kernel='poly', C=100, degree=2)
]
svr_names = ['Linear', 'RBF', 'Polynomial (d=2)']

plt.figure(figsize=(12, 5))
plt.scatter(size, price, c='steelblue', s=30, alpha=0.5, label='Actual', edgecolor='k', zorder=5)

colors = ['red', 'green', 'orange']
for svr, name, color in zip(svr_kernels, svr_names, colors):
    svr.fit(X_house_s, y_house)
    y_pred_plot = svr.predict(X_plot_s)
    r2 = r2_score(y_house, svr.predict(X_house_s))
    plt.plot(X_plot.flatten(), y_pred_plot, color=color, linewidth=2, label=f'{name} (R²={r2:.3f})')

plt.xlabel('Ghar ka Size (sqft)', fontsize=12)
plt.ylabel('Price (Rs)', fontsize=12)
plt.title('SVR — Alag Kernels Compare Karo', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 7: Real World Applications — Har Topic ka Use
---

| Concept | Real World Example | Industry |
|---------|-------------------|----------|
| **SVM Classification** | Email Spam Detection | Tech (Gmail, Outlook) |
| **SVM + RBF Kernel** | Face Recognition | Security, Phones |
| **SVM + Linear Kernel** | Text Classification | NLP, News Categories |
| **SVR** | Stock Price Prediction | Finance |
| **SVM + Soft Margin** | Medical Diagnosis | Healthcare |
| **Kernel Trick** | Handwriting Recognition | OCR Software |

**SVM vs KNN — Kab Kya Use Karein?**

| Feature | SVM | KNN |
|---------|-----|-----|
| Training Speed | Slow (large data) | Fast (kuch nahi) |
| Prediction Speed | Fast | Slow |
| Memory | Small (sirf support vectors) | Large (poora dataset) |
| Non-linear data | Kernel trick se handle | Distance se |
| Best for | High-dimensional data | Small datasets |

In [ ]:
# Real World Example: Text-jaisa high-dimensional data par SVM
from sklearn.datasets import make_classification

# High-dimensional data (20 features — text classification jaisa)
X_hd, y_hd = make_classification(
    n_samples=500, n_features=20, n_informative=10,
    n_classes=2, random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(X_hd, y_hd, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

print("High-Dimensional Data par SVM Performance:")
print(f"Data shape: {X_hd.shape} (500 samples, 20 features)")
print("-" * 45)

for kernel in ['linear', 'rbf', 'poly']:
    svm_hd = SVC(kernel=kernel, C=1.0)
    svm_hd.fit(X_tr, y_tr)
    print(f"{kernel:10}: Train={svm_hd.score(X_tr, y_tr):.3f} | Test={svm_hd.score(X_te, y_te):.3f}")

print("\n✅ SVM linear kernel high-dimensional data par bahut achha kaam karta hai!")

## Notebook Summary Table
---

| Topic | Humne Kya Sikha | Real World Use |
|-------|----------------|----------------|
| SVM Introduction | Maximum margin classifier | Classification, Regression |
| Hyperplane & Margin | Line jo best divide kare | Decision boundary concept |
| Support Vectors | Boundary points jo matter karte hain | Model ka backbone |
| C Parameter | Hard vs Soft Margin trade-off | Regularization tuning |
| Linear Kernel | Linearly separable data | Text classification |
| RBF Kernel | Non-linear, circular boundaries | Face recognition, Biology |
| Polynomial Kernel | Polynomial shaped boundaries | Image processing |
| Gamma Parameter | RBF curve smoothness | Model tuning |
| SVR | Regression with epsilon tube | Price prediction, Forecasting |

---
*Atlas Skilltech University — Machine Learning Course*